<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 16px; padding: 40px; font-family: 'Segoe UI', sans-serif; color: white; margin-bottom: 24px;">
  <div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; color: #94a3b8; margin-bottom: 8px;">Network INAE Program Tutorial — Hands On</div>
  <h1 style="margin: 0 0 12px 0; font-size: 2.2em; font-weight: 700;">🔬 Traffic Capture & Analysis</h1>
  <p style="margin: 0; color: #cbd5e1; font-size: 1.05em; line-height: 1.6;">Capturing live traffic with <code style='background:#1e40af;padding:2px 8px;border-radius:4px;'>tcpdump</code>, inspecting packets with <code style='background:#1e40af;padding:2px 8px;border-radius:4px;'>Wireshark</code>, and programmable analysis using <code style='background:#1e40af;padding:2px 8px;border-radius:4px;'>scapy</code> and <code style='background:#1e40af;padding:2px 8px;border-radius:4px;'>dpkt</code>.</p>
  <hr style="border-color: #334155; margin: 24px 0;">
  <div style="display: flex; gap: 32px; font-size: 0.9em; color: #94a3b8; flex-wrap: wrap;">
    <span>⏱ Estimated Time: ~90 min</span>
    <span>💻 Platform: Linux / macOS</span>
    <span>✍️ Author: Mayank</span>
  </div>
</div>

## 📋 Lab Overview

This notebook progresses through three layers of network traffic work — capture, GUI inspection, and programmatic analysis.

| Section | Tool | What You Do |
|---------|------|-------------|
| 1 | `tcpdump` | Capture live packets from the command line |
| 2 | Wireshark | Inspect raw packets visually in a GUI |
| 3 | scapy | Read and dissect packets programmatically |
| 4 | dpkt + pandas | Parse PCAPs into DataFrames for analysis |
| 5 | Analysis | Explore a real speedtest PCAP |

**Requirements:** A PCAP file (`speedtest.pcap` or any `.pcap`) in a `data/` folder relative to this notebook.
The capture sections (1 & 2) require a machine with a live network interface.

### 📦 Python Requirements

The following libraries are required. Install them from your terminal before running this notebook.

<div style="background: #968f01; border-left: 4px solid #eab308; padding: 16px; border-radius: 8px; font-family: monospace; margin: 8px 0; position: relative;">
<strong>📦 Install required Python libraries (run in terminal):</strong><br><br>
<code>pip install scapy dpkt pandas numpy seaborn</code>
<br><br>
<button onclick="(function(){
  var text = 'pip install scapy dpkt pandas numpy seaborn';
  var ta = document.createElement('textarea');
  ta.value = text; ta.style.position = 'fixed'; ta.style.opacity = '0';
  document.body.appendChild(ta); ta.focus(); ta.select();
  try { document.execCommand('copy'); this.textContent = '✅ Copied!'; } catch(e) { this.textContent = '❌ Failed'; }
  document.body.removeChild(ta);
  setTimeout(function(btn){ btn.textContent = '📋 Copy'; }, 2000, this);
}).call(this)" style="background:#2563eb;color:white;border:none;padding:6px 14px;border-radius:6px;cursor:pointer;font-size:13px;font-weight:600;margin-top:4px;">📋 Copy</button>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0 8px 0;">
  <h2 style="color:#7dd3fc; margin-top:0;">📡 Section 1 — <code style='background:#1e40af;padding:2px 8px;border-radius:4px;font-size:0.85em;'>tcpdump</code>: Capturing Live Traffic</h2>

  <h3 style="color:#93c5fd; margin-bottom:6px;">What is <code>tcpdump</code>?</h3>
  <p style="color:#cbd5e1; margin-bottom:12px;"><code>tcpdump</code> is a command-line packet analyser that captures and inspects live network traffic directly from a network interface. It uses the <strong>libpcap</strong> library to capture raw packets before they are processed by applications or the operating system networking stack.</p>

  <p style="color:#cbd5e1; margin-bottom:12px;">Every packet visible in a browser, game, video call, or SSH session exists first as raw bytes travelling over a wire or wireless medium. <code>tcpdump</code> exposes those packets in real time and allows them to be saved into <code>.pcap</code> files for later analysis using tools such as Wireshark.</p>

  <h3 style="color:#93c5fd; margin-bottom:6px;">How does <code>tcpdump</code> work?</h3>
  <p style="color:#cbd5e1; margin-bottom:8px;"><code>tcpdump</code> listens on a selected network interface and captures packets matching optional filters.</p>

  <pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em; margin:8px 0;">
tcpdump -i eth0
tcpdump -i wlan0 port 80
tcpdump -i any host google.com
  </pre>

  <p style="color:#94a3b8; font-size:0.88em; margin:0;">By default, packets are printed live to the terminal. Using <code>-w</code>, packets can instead be written to a <code>.pcap</code> file for offline inspection.</p>

  <h3 style="color:#93c5fd; margin-bottom:6px;">What does it actually capture?</h3>
  <ul style="color:#94a3b8; margin:0 0 12px 0; padding-left:20px;">
    <li><strong style="color:#e2e8f0;">Ethernet frames</strong> — MAC addresses and link-layer headers</li>
    <li><strong style="color:#e2e8f0;">IP packets</strong> — source/destination IPs and protocols</li>
    <li><strong style="color:#e2e8f0;">TCP/UDP traffic</strong> — ports, flags, sequence numbers</li>
    <li><strong style="color:#e2e8f0;">DNS requests</strong> — domain lookups and responses</li>
    <li><strong style="color:#e2e8f0;">ICMP traffic</strong> — ping requests and replies</li>
  </ul>

  <h3 style="color:#93c5fd; margin-bottom:6px;">Where is <code>tcpdump</code> used in real life?</h3>
  <ul style="color:#94a3b8; margin:0 0 14px 0; padding-left:20px;">
    <li><strong style="color:#e2e8f0;">Network debugging</strong> — engineers inspect live packets to diagnose connectivity problems.</li>
    <li><strong style="color:#e2e8f0;">Security analysis</strong> — analysts detect suspicious traffic, scans, malware communication, or attacks.</li>
    <li><strong style="color:#e2e8f0;">Protocol analysis</strong> — researchers study TCP handshakes, retransmissions, and congestion behavior.</li>
    <li><strong style="color:#e2e8f0;">Application troubleshooting</strong> — verify whether requests are actually reaching servers.</li>
    <li><strong style="color:#e2e8f0;">Packet capture for Wireshark</strong> — collect `.pcap` traces for deeper GUI-based inspection.</li>
  </ul>

  <h3 style="color:#93c5fd; margin-bottom:6px;">Useful Examples</h3>

  <pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em; margin:8px 0;">
# Capture packets on all interfaces
tcpdump -i any

# Capture only ICMP packets (ping)
tcpdump icmp

# Capture DNS traffic
tcpdump port 53

# Save packets to a file
tcpdump -w capture.pcap
  </pre>

  <p style="color:#fbbf24; font-size:0.9em; margin:0;">💡 <strong>Why this is interesting:</strong> Almost everything on the Internet — web requests, video calls, SSH sessions, DNS lookups — ultimately becomes packets. <code>tcpdump</code> lets you observe the Internet at its most fundamental level: raw packets moving across the network in real time.</p>
</div>

### Task 1.1 — Basic Capture

The cells below are **reference commands** — run them in your terminal, not inside the notebook, as they require root/sudo privileges.

| Flag | Meaning |
|------|---------|
| `-i eth0` | Interface to listen on (`any` captures all) |
| `-c 20` | Stop after capturing 20 packets |
| `-s 0` | Capture full packet (default snaplen is 65535 bytes; older versions default to 68) |
| `-n` | Don't resolve hostnames (faster, shows raw IPs) |
| `-v` / `-vv` | Verbose output (more header fields shown) |
| `-w file.pcap` | Write captured packets to file instead of printing |
| `-r file.pcap` | Read from a previously saved pcap file |
| `'filter'` | BPF filter expression — see examples below |

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 The <code>-s</code> Flag — Snaplen</strong><br><br>
The <strong>snapshot length</strong> controls how many bytes of each packet are captured. The default in modern tcpdump is 65535 (full packet). Older systems defaulted to 68 bytes — enough for headers but truncating the payload.
<br><br>
Use <code>-s 0</code> to always capture full packets. If you only need headers (e.g., for flow-level analysis), <code>-s 96</code> captures Ethernet + IP + TCP/UDP headers with no payload, reducing file size significantly.
<br><br>
<strong>Rule of thumb:</strong> Use <code>-s 0</code> for full analysis. Use <code>-s 96</code> when you only care about 5-tuples (src/dst IP, src/dst port, protocol) and want smaller files.
</div>

### Task 1.2 — Common tcpdump Commands

<div style="background:#968f01; border-left: 4px solid #eab308; padding: 16px; border-radius: 8px; font-family: monospace; margin: 8px 0;">
<strong>Run in terminal — basic captures:</strong>
</div>

In [ ]:
# Show available interfaces on your machine
!tcpdump -D

In [ ]:
# Capture 10 packets on any interface — print to screen
# Remove 'sudo' if running as root; add it otherwise
!sudo tcpdump -i any -c 10 -n

In [ ]:
# Capture 50 full packets and save to a pcap file
# -s 0 ensures full packet capture (not truncated)
!sudo tcpdump -i any -c 50 -s 0 -w data/capture.pcap

In [ ]:
# Read and display packets from the saved file
!tcpdump -r data/capture.pcap -n -v | head -30

### Task 1.3 — BPF Filters

`tcpdump` uses **Berkeley Packet Filter (BPF)** expressions to select which packets to capture. Filters are evaluated in the kernel, so unmatched packets never reach userspace — this is efficient even on high-traffic links.

```bash
# Only TCP traffic
sudo tcpdump -i any -c 20 'tcp'

# Only traffic to/from port 443 (HTTPS)
sudo tcpdump -i any -c 20 'port 443'

# Only traffic from a specific host
sudo tcpdump -i any -c 20 'src host 8.8.8.8'

# TCP SYN packets only (connection initiations)
sudo tcpdump -i any -c 20 'tcp[tcpflags] & tcp-syn != 0'

# Capture DNS (port 53) queries — useful for observing name resolution
sudo tcpdump -i any -c 20 -s 0 -w data/dns.pcap 'port 53'
```

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 1</strong>
<ol>
<li>Run <code>tcpdump -D</code>. How many interfaces does your machine expose? What is the <code>any</code> pseudo-interface?</li>
<li>What is the difference between capturing with <code>-s 0</code> and the default snaplen? In what scenario would you deliberately use a smaller snaplen?</li>
<li>Capture 20 packets with no filter, then 20 packets with <code>'port 443'</code>. How does the output differ?</li>
<li>tcpdump requires root/sudo on most systems. Why? What security risk does unrestricted packet capture pose?</li>
</ol>
</div>

**✍️ Your Observations (Section 1):**

```
Interfaces found     : 
Packets without filter (protocol mix)  : 
Packets with port 443 filter           : 
```

*Analysis:*

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0 8px 0;">
  <h2 style="color:#7dd3fc; margin-top:0;">🦈 Section 2 — <code style='background:#1e40af;padding:2px 8px;border-radius:4px;font-size:0.85em;'>Wireshark</code>: GUI-Based Raw Packet Viewing</h2>

  <h3 style="color:#93c5fd; margin-bottom:6px;">What is <code>Wireshark</code>?</h3>
  <p style="color:#cbd5e1; margin-bottom:12px;"><strong>Wireshark</strong> is a graphical packet analyser that captures and visualizes network traffic packet-by-packet. It reads <code>.pcap</code> files (often generated using <code>tcpdump</code>) and decodes protocols layer-by-layer:</p>

  <pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em; margin:8px 0;">
Ethernet → IP → TCP/UDP → Application
  </pre>

  <p style="color:#cbd5e1; margin-bottom:12px;">Wireshark allows you to inspect individual packets, headers, flags, payloads, retransmissions, DNS lookups, TCP handshakes, TLS exchanges, and much more using an interactive GUI.</p>

  <h3 style="color:#93c5fd; margin-bottom:6px;">Where is <code>Wireshark</code> used?</h3>
  <ul style="color:#94a3b8; margin:0 0 14px 0; padding-left:20px;">
    <li><strong style="color:#e2e8f0;">Protocol debugging</strong> — inspect exactly what a client or server transmitted.</li>
    <li><strong style="color:#e2e8f0;">Classroom demonstrations</strong> — visualize TCP handshakes, DNS queries, TLS negotiation, and HTTP traffic.</li>
    <li><strong style="color:#e2e8f0;">Performance troubleshooting</strong> — detect retransmissions, resets (RSTs), and packet loss.</li>
    <li><strong style="color:#e2e8f0;">Security analysis</strong> — inspect suspicious traffic or malformed packets.</li>
  </ul>

  <p style="color:#fbbf24; font-size:0.9em; margin:0;">💡 <strong>Why this is interesting:</strong> Wireshark transforms invisible network traffic into something humans can visually inspect and reason about — making it one of the most important tools in networking, systems, and cybersecurity.</p>
</div>

### Workflow: tcpdump → Wireshark

A common pattern: capture headlessly on a server with `tcpdump`, copy the `.pcap` to your laptop, open in Wireshark.

```bash
# 1. Capture on remote server
sudo tcpdump -i eth0 -c 500 -s 0 -w /tmp/session.pcap

# 2. Open in Wireshark (or from command line)
wireshark ~/Desktop/session.pcap
```

### Key Wireshark Features to Know

| Feature | How to use | What it shows |
|---------|-----------|---------------|
| **Display filter** | Top bar: `tcp.port == 443` | Only HTTPS packets |
| **Follow TCP stream** | Right-click a packet → Follow → TCP Stream | Full conversation between two endpoints |
| **Protocol hierarchy** | Statistics → Protocol Hierarchy | Breakdown of protocols by packet count |
| **Conversations** | Statistics → Conversations | All flows sorted by bytes/packets |
| **IO Graph** | Statistics → IO Graph | Throughput over time |
| **Expert info** | Analyze → Expert Information | All retransmissions, resets, warnings |

### Useful Display Filters

```
tcp.flags.syn == 1 && tcp.flags.ack == 0   # SYN-only: new connection starts
tcp.flags.rst == 1                          # RST: aborted connections
http                                        # All HTTP traffic
dns                                         # All DNS queries and replies
ip.addr == 8.8.8.8                         # Any traffic to/from Google DNS
tcp.analysis.retransmission                # Retransmitted TCP segments
```

<img src="images/image (3).png" width="100%">

<br><br>

<img src="images/image (5).png" width="100%">

<br><br>

<img src="images/image (6).png" width="100%">

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 2</strong><br>
Open your captured <code>data/capture.pcap</code> (or <code>speedtest.pcap</code>) in Wireshark and answer:
<ol>
<li>Go to <strong>Statistics → Protocol Hierarchy</strong>. What are the top 3 protocols by packet count?</li>
<li>Find a TCP packet. Click on it. In the packet details pane, expand the TCP layer. What is the <strong>sequence number</strong>, <strong>ACK number</strong>, and <strong>window size</strong>?</li>
<li>Right-click on a TCP packet and select <strong>Follow → TCP Stream</strong>. What application-layer data can you see? Is any of it human-readable?</li>
<li>Apply the filter <code>tcp.analysis.retransmission</code>. How many retransmissions are in your capture? What does each one indicate?</li>
</ol>
</div>

**✍️ Your Observations (Section 2):**

```
Top protocols (by packet count)   : 1.  2.  3.
TCP retransmissions found         : 
```

*TCP segment fields observed:*

*What did the TCP stream show?*

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0 8px 0;">
  <h2 style="color:#7dd3fc; margin-top:0;">🐍 Section 3 — <code style='background:#1e40af;padding:2px 8px;border-radius:4px;font-size:0.85em;'>scapy</code>: Programmatic Packet Dissection</h2>

  <h3 style="color:#93c5fd; margin-bottom:6px;">What is <code>scapy</code>?</h3>
  <p style="color:#cbd5e1; margin-bottom:12px;"><code>scapy</code> is a Python library for packet manipulation, packet crafting, and protocol analysis. It can read <code>.pcap</code> files, dissect packets layer-by-layer, generate custom packets from scratch, and even transmit them over the network.</p>

  <p style="color:#cbd5e1; margin-bottom:12px;">For analysis, <code>rdpcap()</code> loads packets into Python as packet objects. Each packet behaves like a stack of protocol layers:</p>

  <pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em; margin:8px 0;">
Ethernet → IP → TCP/UDP → Application
  </pre>

  <p style="color:#94a3b8; font-size:0.88em; margin:0 0 14px 0;">Individual layers can be accessed using:</p>

  <pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em; margin:8px 0;">
pkt[IP]
pkt[TCP]
pkt.payload
  </pre>

  <h3 style="color:#93c5fd; margin-bottom:6px;">Where is <code>scapy</code> used?</h3>
  <ul style="color:#94a3b8; margin:0 0 14px 0; padding-left:20px;">
    <li><strong style="color:#e2e8f0;">Packet analysis</strong> — inspect headers, flags, payloads, and protocol behavior programmatically.</li>
    <li><strong style="color:#e2e8f0;">Security research</strong> — craft custom packets for testing firewalls, IDS systems, or protocol robustness.</li>
    <li><strong style="color:#e2e8f0;">Networking experiments</strong> — automate traffic generation and protocol measurements.</li>
    <li><strong style="color:#e2e8f0;">Education</strong> — visualize and manipulate packets directly in Python notebooks.</li>
  </ul>

  <p style="color:#fbbf24; font-size:0.9em; margin:0;">💡 <strong>Why this is interesting:</strong> Unlike tools such as <code>tcpdump</code> or Wireshark that mainly observe packets, <code>scapy</code> allows you to treat packets as programmable Python objects — inspect them, modify them, generate them, and automate entire network experiments.</p>
</div>

In [ ]:
# ── All imports — run this cell first before any other ──────────────────
import socket
import binascii

# Packet analysis
from scapy.all import rdpcap, wrpcap, PacketList
from scapy.all import Ether, IP, IPv6, TCP, UDP, ICMP

# Data & numerics
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(color_codes=True)

print("\u2705 All libraries imported successfully.")

### Task 3.1 — Reading a PCAP with scapy

In [ ]:
# Load a pcap file — returns a PacketList object
pcap = rdpcap('data/speedtest.pcap')
print(pcap)

In [ ]:
# Summary of packets in the capture
print(f"Total packets: {len(pcap)}")
print(f"\nFirst 5 packet summaries:")
for pkt in pcap[:5]:
    print(" ", pkt.summary())

In [ ]:
# Inspect one packet in detail — .show() prints all layers and fields
sample_pkt = pcap[2446]
sample_pkt.show()

### Task 3.2 — Ethernet Layer

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 Ethernet Frame Format</strong>
<pre style="margin: 8px 0 0 0; font-size: 0.88em;">
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    Destination MAC Address (6B)                |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                      Source MAC Address (6B)                   |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|    EtherType (2B)    |           Payload (46–1500B)            |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|          Frame Check Sequence (4B)           |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
</pre>
<br>
<strong>MAC address</strong> — a 48-bit hardware identifier, unique per NIC. Operates at Layer 2 (Data Link). Unlike IP addresses, MACs are not routed beyond the local network segment.<br><br>
<strong>EtherType</strong> — identifies the encapsulated protocol: <code>0x0800</code> = IPv4, <code>0x86DD</code> = IPv6, <code>0x0806</code> = ARP.
</div>

In [ ]:
ethernet_frame = sample_pkt

print(f"Type      : {type(ethernet_frame)}")
print(f"Timestamp : {ethernet_frame.time}  (Unix epoch)")
print(f"Src MAC   : {ethernet_frame.src}")
print(f"Dst MAC   : {ethernet_frame.dst}")
print(f"EtherType : {hex(ethernet_frame.type)}")

### Task 3.3 — IP Layer

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 IPv4 Header Format</strong>
<pre style="margin: 8px 0 0 0; font-size: 0.88em;">
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|Version|  IHL  |Type of Service|          Total Length         |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|         Identification        |Flags|      Fragment Offset    |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|  Time to Live |    Protocol   |         Header Checksum       |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                       Source Address (32 bits)                 |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    Destination Address (32 bits)               |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
</pre>
<br>
Key fields: <strong>TTL</strong> — decremented by each router (traceroute exploits this). <strong>Protocol</strong> — identifies L4: 6=TCP, 17=UDP, 1=ICMP. <strong>Total Length</strong> — full packet size in bytes.
</div>

In [ ]:
ip_pkt = ethernet_frame.payload   # or: ethernet_frame[IP]

print(f"Source IP   : {ip_pkt.src}")
print(f"Dest IP     : {ip_pkt.dst}")
print(f"TTL         : {ip_pkt.ttl}")
print(f"Protocol    : {ip_pkt.proto}  (6=TCP, 17=UDP, 1=ICMP)")
print(f"Total len   : {ip_pkt.len} bytes")

### Task 3.4 — Transport Layer: TCP and UDP

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 TCP Header (key fields)</strong>
<pre style="margin: 8px 0 0 0; font-size: 0.88em;">
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|          Source Port          |       Destination Port        |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                        Sequence Number                        |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    Acknowledgment Number                      |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
| Offset|  Res  |  Flags (SYN,ACK,FIN,RST,PSH,URG)  |  Window |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
</pre>
<br>
<strong>UDP Header (4 fields only):</strong> Source Port | Destination Port | Length | Checksum — no seq/ack, no retransmission, no flow control.
</div>

In [ ]:
# TCP segment from sample_pkt
transport_segment = ip_pkt.payload
print(f"Layer type  : {type(transport_segment).__name__}")
transport_segment.show()

In [ ]:
# Access individual TCP fields
if transport_segment.name == 'TCP':
    tcp = transport_segment
    print(f"Src port   : {tcp.sport}")
    print(f"Dst port   : {tcp.dport}")
    print(f"Seq        : {tcp.seq}")
    print(f"Ack        : {tcp.ack}")
    print(f"Flags      : {tcp.flags}  (S=SYN, A=ACK, F=FIN, R=RST, P=PSH)")
    print(f"Window     : {tcp.window} bytes")

In [ ]:
# Find a UDP packet in the pcap
udp_pkts = [p for p in pcap if UDP in p]
print(f"UDP packets found: {len(udp_pkts)}")

if udp_pkts:
    udp_sample = udp_pkts[0]
    udp_layer = udp_sample[UDP]
    print(f"\nUDP sample — src:{udp_layer.sport} → dst:{udp_layer.dport}  len:{udp_layer.len}")

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 3</strong>
<ol>
<li>What EtherType value does your sample packet show? What does that value mean?</li>
<li>Look at <code>ip_pkt.ttl</code>. What does the TTL value tell you about how far the packet has already travelled? (Consider: typical initial TTL values are 64 or 128.)</li>

<li>How many UDP packets were in your capture vs TCP?</li>
</ol>
</div>

**✍️ Your Observations (Section 3):**

```
EtherType value      : 
IP TTL observed      : 
TCP flags in sample  : 
TCP packet count     : 
UDP packet count     : 
```

*Analysis:*

---
# 🐼 Section 4 — PCAP to DataFrame using scapy

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 Why scapy + pandas?</strong><br><br>
<code>scapy</code> is excellent for per-packet inspection but is slow over large captures. <code>dpkt</code> is a leaner parser — it extracts header fields quickly without building rich Python objects for every layer. Combined with <code>pandas</code>, you can analyse tens of thousands of packets as a DataFrame: filter, group, aggregate, and plot with familiar data-science tooling.
<br><br>
The workflow: <code>dpkt</code> parses the pcap → Python extracts fields per packet → pandas builds a DataFrame → you analyse it like any tabular dataset.
</div>

In [ ]:
# ── Field name lists ──────────────────────────────────────────────
ip_fields  = ['src_ip', 'dst_ip', 'len', 'proto']
tcp_fields = ['sport', 'dport', 'seq', 'ack',
              'dataofs', 'flags', 'window',
              'chksum', 'urgptr', 'options', 'payload_len']

udp_fields = ['sport', 'dport', 'len', 'chksum']

# ── Per-layer extraction functions ──────────────────────────────

def get_ip4_fields(pkt):
    ip = pkt[IP]
    return [ip.src,ip.dst,ip.len,ip.proto]


def get_ip6_fields(pkt):
    ip6 = pkt[IPv6]
    return [ip6.src,ip6.dst,ip6.plen,ip6.nh]


def get_tcp_fields(pkt):
    tcp = pkt[TCP]
    return [tcp.sport,tcp.dport,tcp.seq,tcp.ack,tcp.dataofs,tcp.flags,
            tcp.window,tcp.chksum,tcp.urgptr,tcp.options,len(bytes(tcp.payload))]


def get_udp_fields(pkt):
    udp = pkt[UDP]
    return [udp.sport,udp.dport,udp.len,udp.chksum]


print("✅ Scapy helper functions defined.")

In [ ]:
def pcap_to_df(file_path):
    """
    Parse a pcap file into a pandas DataFrame
    with one row per packet using Scapy.
    """
    rows = []
    packets = rdpcap(file_path)
    for pkt in packets:
        try:
            # ── IPv4 ─────────────────────────────
            if IP in pkt:
                ip_row = get_ip4_fields(pkt)
                proto = pkt[IP].proto
            # ── IPv6 ────────────────────────────
            elif IPv6 in pkt:
                ip_row = get_ip6_fields(pkt)
                proto = pkt[IPv6].nh
            else:
                continue
        except Exception:
            continue
        dummy_tcp = [None] * len(tcp_fields)
        dummy_udp = [None] * len(udp_fields)
        # ── UDP ────────────────────────────────
        if proto == 17 and UDP in pkt:
            transport = get_udp_fields(pkt) + dummy_tcp
        # ── TCP ────────────────────────────────
        elif proto == 6 and TCP in pkt:
            transport = dummy_udp + get_tcp_fields(pkt)
        else:
            continue
        rows.append(
            [pkt.time] + ip_row + transport
        )
    cols = (
        ['ts']
        + ip_fields
        + [f'udp_{c}' for c in udp_fields]
        + [f'tcp_{c}' for c in tcp_fields]
    )
    return pd.DataFrame(rows, columns=cols)


print("✅ Scapy pcap_to_df() defined.")

In [ ]:
# Load the pcap into a DataFrame
file_path = './data/speedtest.pcap'
df = pcap_to_df(file_path)
print(f"Shape: {df.shape}  → {df.shape[0]:,} packets, {df.shape[1]} columns")
df.head()

### Task 4.1 — DataFrame Basics

In [ ]:
# First and last 5 rows
display(df.head())
display(df.tail())

In [ ]:
# Column types and null counts
df.info()

In [ ]:
# Select key columns — the 5-tuple + protocol
df[['src_ip', 'dst_ip', 'udp_sport', 'udp_dport', 'tcp_sport', 'tcp_dport', 'proto']].head(10)

In [ ]:
# Protocol distribution: 6=TCP, 17=UDP
proto_map = {6: 'TCP', 17: 'UDP'}
df['proto_name'] = df['proto'].map(proto_map)
print(df['proto_name'].value_counts())

---
# 📈 Section 5 — PCAP Throughput and Packet Size Analysis

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; padding: 16px; border-radius: 8px; margin: 12px 0;">
<strong>🔵 Concept Refresher — How does a speed test work?</strong><br><br>

A speed test estimates the available network bandwidth by temporarily <strong>saturating the connection</strong>. The client and server exchange traffic as aggressively as possible over a short time window, attempting to fully utilize the network path.

The measured <strong>throughput</strong> is then computed from the amount of data successfully transferred during the test interval:

<br><br>

<div align="center">

Throughput = Data Transferred / Time

</div>

<br>

In a packet capture (PCAP), this behavior becomes visible as:
<ul>
<li>A dominant client–server IP pair carrying most of the traffic</li>
<li>Sustained bursts of TCP or UDP packets</li>
<li>High packet rates and larger aggregate byte counts</li>
<li>Temporal fluctuations in throughput caused by congestion control and network conditions</li>
</ul>

By aggregating packet sizes over time, we can reconstruct the throughput profile of the speed test and observe how the network behaves during link saturation.
</div>

### Task 5.1 — Find the Speed Test Server

In [ ]:
# Fill NaN with 0 (TCP and UDP fields are mutually exclusive per row)
df = df.fillna(0)

# Group by source IP + port + protocol, sum total bytes sent
df_flows = (
    df.groupby(['src_ip', 'udp_sport', 'tcp_sport', 'proto'])
    .agg(total_bytes=('len', 'sum'), packet_count=('len', 'count'))
    .reset_index()
    .sort_values('total_bytes', ascending=False)
)

print("Top 10 flows by bytes:")
df_flows.head(10)

In [ ]:
# Top source IPs by total bytes — these are the dominant talkers
df_top = (
    df.groupby('src_ip')
    .agg(total_bytes=('len', 'sum'), packets=('len', 'count'))
    .reset_index()
    .sort_values('total_bytes', ascending=False)
    .head(10)
)
df_top['total_MB'] = (df_top['total_bytes'] / 1e6).round(2)
print(df_top.to_string(index=False))

### Task 5.2 — Separate Inbound and Outbound Traffic

5.2  **✍️ Identify the Local Client IP :**


In [ ]:
# Replace with the local/client IP you identified above
LOCAL_IP =    # e.g. '192.168.1.5' or the IPv6 address

df['direction'] = df['src_ip'].apply(
    lambda src: 'outbound' if src == LOCAL_IP else 'inbound'
)

summary = df.groupby('direction').agg(
    total_bytes=('len', 'sum'),
    packets=('len', 'count')
).reset_index()
summary['total_MB'] = (summary['total_bytes'] / 1e6).round(2)
print(summary.to_string(index=False))

### Task 5.3 — Throughput Over Time

In [ ]:
def plot_throughput(
    df,
    granularity_sec=1.0,
    throughput_unit='Mbps'
):
    """
    Plot throughput with configurable time granularity.

    Parameters
    ----------
    df : pandas.DataFrame
        Packet dataframe.

    granularity_sec : float
        Time bucket size in seconds.
        Examples:
            1.0  -> per second
            0.1  -> 100 ms
            5.0  -> 5 second bins

    throughput_unit : str
        'bps', 'Kbps', 'Mbps', 'Gbps'
    """

    df = df.copy()

    # ── Create time buckets ─────────────────────────────
    df['ts_bucket'] = (
        (df['ts'] / granularity_sec)
        .astype(int)
    )

    # ── Aggregate throughput ────────────────────────────
    df_tput = (
        df.groupby(['ts_bucket', 'direction'])
        .agg(bytes_per_bucket=('len', 'sum'))
        .reset_index()
    )

    # ── Convert to bits/sec ─────────────────────────────
    bits_per_sec = (
        df_tput['bytes_per_bucket'] * 8
        / granularity_sec
    )

    # ── Unit scaling ────────────────────────────────────
    scale_map = {
        'bps': 1,
        'Kbps': 1e3,
        'Mbps': 1e6,
        'Gbps': 1e9
    }

    scale = scale_map[throughput_unit]

    df_tput['throughput'] = bits_per_sec / scale

    # ── Normalised time axis ────────────────────────────
    t0 = df_tput['ts_bucket'].min()

    df_tput['t'] = (
        (df_tput['ts_bucket'] - t0)
        * granularity_sec
    )

    # ── Plot ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 4))

    for direction, grp in df_tput.groupby('direction'):

        ax.plot(
            grp['t'],
            grp['throughput'],
            label=direction
        )

    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'Throughput ({throughput_unit})')

    ax.set_title(
        f'Throughput ({granularity_sec}s granularity)'
    )

    ax.legend()

    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return df_tput

In [ ]:
per_second = plot_throughput(df)

In [ ]:
## Plot the graph for smaller granularity say 0.25 seconds

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Task 5.3</strong>
<ol>
<li>Look at the <strong>1-second granularity</strong> throughput plot. Does the download curve ramp up gradually at the start? What TCP mechanism causes this shape, and roughly how many seconds does it take to plateau?</li>
<li>Compare the 1-second and 0.25-second granularity plots. What detail does finer granularity reveal (bursts, gaps)? Why might per-second averaging hide important behaviour?</li>
<li>Note that this plot includes <strong>all traffic</strong> in the capture, not just the speed test flow. What other flows might be contributing noise? How would you isolate only the speed-test server traffic? (This is exactly what Section 6 does.)</li>
</ol>
</div>

In [ ]:
# Packet size distribution
fig, ax = plt.subplots(figsize=(10, 4))
df['len'].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Packet size (bytes)')
ax.set_ylabel('Count')
ax.set_title('Packet Size Distribution')
ax.axvline(1460, color='red', linestyle='--', label='Typical TCP MSS (1460B)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean packet size : {df['len'].mean():.0f} bytes")
print(f"Median           : {df['len'].median():.0f} bytes")
print(f"Max              : {df['len'].max()} bytes")

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 5</strong>
<ol>
<li>Which IP address sent the most bytes? Is it the client or the speed test server? How did you determine which direction was the download vs upload?</li>
<li>Look at the throughput-over-time plot. Does throughput ramp up at the start? Does it plateau? Explain what network mechanism produces this shape. (Hint: TCP slow start, then congestion avoidance.)</li>
<li>Look at the packet size distribution. Why is there a spike near 1460 bytes? What is the <strong>MSS (Maximum Segment Size)</strong> and why is it typically 1460 bytes for Ethernet?</li>
<li>Are there many small packets (< 100 bytes) in the capture? What do these likely represent? (Hint: TCP ACKs, control messages.)</li>
</ol>
</div>

**✍️ Your Observations (Section 5):**

```
Download throughput peak     :  Mbps
Upload throughput peak       :  Mbps
Most common packet size      :  bytes
```

*Throughput ramp-up explanation:*

*Why 1460 bytes?*

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0 8px 0;">
<h2 style="color:#7dd3fc; margin-top:0;">🚀 Section 6 — Speed Test Measurement</h2>
<h3 style="color:#93c5fd; margin-bottom:6px;">Throughput vs Speed Test — what is the difference?</h3>
<p style="color:#cbd5e1; margin-bottom:12px;">
The throughput plots in Task 5.3 show <strong>all traffic</strong> in the capture — background DNS queries, TCP control packets, and any concurrent flows. A <strong>speed-test measurement</strong> is different: it isolates the single IP pair between your machine and the speed-test server, so you see only the intentional bulk transfer and can read off the true upload and download capacity of your link without noise.
</p>
<p style="color:#cbd5e1; margin-bottom:0;">
The workflow: <strong>(1) identify the dominant IP pair → (2) filter the DataFrame to that pair → (3) label direction (download vs upload) → (4) compute per-second throughput → (5) plot.</strong>
</p>
</div>

In [ ]:
# ── Step 1: Identify the speed-test server IP ──────────────────────────────
# The server is the remote IP that accounts for the bulk of bytes.
# In a typical 20-second test the server sends >> 10 MB in the download direction.

df_pairs = (
    df.groupby(['src_ip', 'dst_ip','udp_sport','udp_dport','tcp_sport', 'tcp_dport'])
    .agg(total_bytes=('len', 'sum'), packets=('len', 'count'))
    .reset_index()
    .sort_values('total_bytes', ascending=False)
)
df_pairs['total_MB'] = (df_pairs['total_bytes'] / 1e6).round(2)

print('Top 10 IP pairs by bytes transferred:')
print(df_pairs.head(10).to_string(index=False))



**✍️ Identify the Speed-Test Server(s):**


Candidate server IP(s):

1.

2.

3.

Dominant protocol : TCP / UDP

Common server port: 443


In [ ]:
# The two dominant rows should be the same pair in reverse order:
#   server → client   (download — large byte count)
#   client → server   (upload   — smaller byte count)
# Identify the speedtest server
SERVER_IP = ""
print(f'\nSelected server IP : {SERVER_IP}')

In [ ]:
# ── Step 2: Filter to only the speed-test flow ──────────────────────────────
# SERVER_IP = "x.x.x.x"   # ← uncomment and set manually if needed

df_st = df[
    (df['src_ip'] == SERVER_IP) | (df['dst_ip'] == SERVER_IP)
].copy()

df_st['direction'] = df_st['src_ip'].apply(
    lambda s: 'download' if s == SERVER_IP else 'upload'
)

n_down  = (df_st['direction'] == 'download').sum()
n_up    = (df_st['direction'] == 'upload').sum()
mb_down = df_st.loc[df_st['direction'] == 'download', 'len'].sum() / 1e6
mb_up   = df_st.loc[df_st['direction'] == 'upload',   'len'].sum() / 1e6

print(f'Speed-test packets : {len(df_st):,}  '
      f'({n_down:,} download / {n_up:,} upload)')
print(f'Data volume        : {mb_down:.1f} MB download  |  {mb_up:.1f} MB upload')
print(f'Server IP          : {SERVER_IP}')

In [ ]:
# ── Throughput calculation ────────────────────────────────────────

def compute_speedtest_throughput(df_st, granularity_sec=1.0, unit='Mbps'):
    scale = {'bps':1, 'Kbps':1e3, 'Mbps':1e6, 'Gbps':1e9}[unit]
    df_w = df_st.copy()
    df_w['bucket'] = (df_w['ts'] / granularity_sec).astype(int)
    agg = (
        df_w.groupby(['bucket', 'direction'])
        .agg(bytes_sum=('len', 'sum'))
        .reset_index()
    )
    agg['throughput'] = agg['bytes_sum'] * 8 / granularity_sec / scale
    t0 = agg['bucket'].min()
    agg['t'] = (agg['bucket'] - t0) * granularity_sec
    print(f'{"Direction":>10} {"Peak":>10} {"Avg":>10} {"Median":>10}')
    for direction in ['download', 'upload']:
        grp = agg[agg['direction'] == direction]['throughput']
        if len(grp):
            print(f'{direction:>10} {grp.max():>8.1f} {unit} {grp.mean():>8.1f} {unit} {grp.median():>8.1f} {unit}')
    return agg


# ── Plotting ──────────────────────────────────────────────────────

def plot_direction_throughput(agg, direction, granularity_sec=1.0, unit='Mbps'):
    grp = agg[agg['direction'] == direction]

    if grp.empty:
        print(f'No {direction} traffic found.')
        return

    avg = grp['throughput'].mean()
    colour = {'download':'#3b82f6', 'upload':'#f59e0b'}[direction]

    plt.figure(figsize=(12,4))
    plt.fill_between(grp['t'], grp['throughput'], alpha=0.2, color=colour)
    plt.plot(grp['t'], grp['throughput'], color=colour, linewidth=1.5)
    plt.axhline(avg, linestyle='--', linewidth=1.2, alpha=0.8,
                color=colour, label=f'avg {avg:.1f} {unit}')

    plt.xlabel('Time (s)')
    plt.ylabel(f'Throughput ({unit})')
    plt.title(f'Speed Test {direction.capitalize()} Throughput ({granularity_sec}s bins)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── Plot the graph for 250ms ─────────────────────────────────────────────────


<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 6</strong>
<ol>
<li>Compare the speed-test download plot (Section 6) against the all-traffic throughput in Task 5.3. Are they identical? What extra traffic was included in Task 5.3 that is now filtered out?</li>

<li>What is the peak download throughput from the speed-test plot? How does it compare to your subscribed ISP speed tier? Name two reasons the measured value may be lower.</li>

<li>Is the reported throughput the true network speed-test result? In this analysis, throughput is averaged over aggregation windows rather than measured exactly over the original speed-test application interval. How can the choice of granularity/window size affect the observed throughput?</li>

<li>Do we observe any noticeable behavioural differences between TCP and UDP traffic in the capture? Compare aspects such as throughput stability, packet sizes, retransmissions, burstiness, or congestion-control behaviour.</li>
</ol>
</div>

**✍️ Your Observations (Section 6 — Speed Test):**

```
Speed-test server IP         : 
Download — peak throughput   :  Mbps
Download — avg throughput    :  Mbps
Upload   — peak throughput   :  Mbps
Upload   — avg throughput    :  Mbps
Ramp-up duration (approx.)   :  seconds
Download / Upload ratio      : 
```

*How does the speed-test plot differ from the all-traffic plot in Task 5.3?*

*Ramp-up explanation (TCP slow-start):*

*Is the link symmetric?*

---
## ✅ Summary & Key Takeaways

<div style="background:#0f172a; color: #e2e8f0; border-radius: 12px; padding: 24px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
<h3 style="color: #7dd3fc; margin-top: 0;">The Traffic Analysis Toolkit</h3>
<table style="width:100%; border-collapse: collapse;">
  <tr style="border-bottom: 1px solid #334155;">
    <th style="text-align:left; padding: 10px; color: #94a3b8;">Tool</th>
    <th style="text-align:left; padding: 10px; color: #94a3b8;">Use when</th>
    <th style="text-align:left; padding: 10px; color: #94a3b8;">Strength</th>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>tcpdump</code></td>
    <td style="padding: 10px;">Headless/server capture, scripting</td>
    <td style="padding: 10px;">Lightweight, fast, BPF filters, works everywhere</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;">Wireshark</td>
    <td style="padding: 10px;">Interactive investigation, debugging, teaching</td>
    <td style="padding: 10px;">Visual, protocol-aware, follow-stream, expert info</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;">scapy</td>
    <td style="padding: 10px;">Per-packet logic, crafting, scripted inspection</td>
    <td style="padding: 10px;">Pythonic, full layer access, can craft + send packets</td>
  </tr>
  <tr>
    <td style="padding: 10px; color: #7dd3fc;">dpkt + pandas</td>
    <td style="padding: 10px;">Large-scale analysis, statistics, plots</td>
    <td style="padding: 10px;">Fast parsing, DataFrame operations, visualisation</td>
  </tr>
</table>
<p style="color: #94a3b8; margin: 16px 0 0 0;">The progression: <strong style="color:#7dd3fc;">Capture → View → Dissect → Analyse</strong> — each tool adds a layer of insight.</p>
</div>

---
# 📎 Appendix — Extended Questions for Students

### A.1 — tcpdump

**A.** Run `sudo tcpdump -i any -c 100 -s 0 -w capture.pcap`. Then run `tcpdump -r capture.pcap -n | wc -l`. How many lines of output are produced per packet on average? What does each line represent?

**B.** Explain the `-s` flag. What is the default snaplen in modern tcpdump? If you capture with `-s 96`, what information is preserved and what is lost? Give a use case where `-s 96` is preferable to `-s 0`.

**C.** Write a BPF filter expression to capture only TCP SYN packets (new connection initiations) destined for port 80 or 443. Explain each part of your filter.

**D.** tcpdump requires root or membership of the `pcap` group on Linux. Why is unrestricted packet capture considered a security risk? What could a malicious user learn from a capture on a shared network?

**E.** Run `sudo tcpdump -i any -c 50 port 53` while browsing a new website. Attach the output. How many DNS queries do you see for a single page load? What does this reveal about how web pages are structured?

### A.2 — Wireshark

**A.** Open a pcap in Wireshark. Use **Statistics → Protocol Hierarchy** to list the top 5 protocols by percentage of packets. Are the percentages what you expected given the capture scenario?

**B.** Find a TCP three-way handshake (SYN, SYN-ACK, ACK) in the capture. For each of the three packets, record: timestamp, source IP:port, destination IP:port, TCP flags, sequence number, acknowledgment number.

**C.** Apply the display filter `tcp.analysis.retransmission`. How many retransmissions are present? For each, identify whether it is a full retransmission or a fast retransmit. What does each indicate?

**D.** Use **Statistics → Conversations → TCP** tab. Sort by bytes. What is the largest single TCP conversation in bytes? What ports are involved, and what application does that likely correspond to?

**E.** Use **Statistics → IO Graph**. Set the Y-axis to bits/second. Describe the throughput shape over time. Does it ramp up slowly? Is it bursty or steady? Relate this to TCP's congestion control mechanisms.

### A.3 — scapy & dpkt Analysis

**A.** Using scapy, write a Python snippet that iterates over all packets in a pcap and prints the 5-tuple (src IP, dst IP, src port, dst port, protocol) for every TCP packet. How many unique 5-tuples (flows) are in the capture?

**B.** What is the TTL value of packets from the speed test server? Given that initial TTL values are typically 64 (Linux) or 128 (Windows), how many hops away is the server?

**C.** From the DataFrame, compute the **average packet size** for TCP vs UDP packets separately. Why does your result make sense given the applications typically using each protocol?

**D.** Reconstruct the per-second throughput graph for inbound and outbound traffic separately. At which second does the download throughput peak? How does this compare to the nominal link speed you know?

**E.** Using dpkt or scapy, count the number of TCP packets with the SYN flag set. Each one represents a new connection attempt. How many connections were opened during this capture? What does this tell you about the application's connection behaviour?

# Answers

In [ ]:
#Section 6

agg_250ms = compute_speedtest_throughput(df_st, granularity_sec=0.25)

plot_direction_throughput(agg_250ms, 'download', granularity_sec=0.25)
plot_direction_throughput(agg_250ms, 'upload', granularity_sec=0.25)

---
## 📚 Reference & Further Reading

| Resource | Link |
|----------|------|
| tcpdump man page | `man tcpdump` |
| tcpdump filter syntax (BPF) | https://www.tcpdump.org/manpages/pcap-filter.7.html |
| Wireshark display filter reference | https://wiki.wireshark.org/DisplayFilters |
| scapy documentation | https://scapy.readthedocs.io |
| dpkt on PyPI | https://pypi.org/project/dpkt |
| Wireshark sample captures | https://wiki.wireshark.org/SampleCaptures |
| RFC 793 — TCP | https://www.rfc-editor.org/rfc/rfc793 |
| RFC 768 — UDP | https://www.rfc-editor.org/rfc/rfc768 |

---
<sub>Notebook authored by **Mayank** with help of Claude and ChatGPT .</sub>
<sub>Parts of notebook borrowed from coursework COL7560, Semester 2 2025-26, by Professor Tarun Mangla · Network INAE Program · 2026</sub>